# Avance 2: Lectura, perfilamiento y preparación de datos con Python

## Proyecto: Análisis de factores de éxito comercial en la industria de videojuegos

**Equipo:** Risk Analysts

**Alumno:** Hernández Martínez Ricardo

**Asignatura:** Inteligencia de negocios (BI)

**Repositorio:** https://github.com/Richi0001/proyecto_Bi_Videojuegos

## 1. Continuidad con el Avance 1

**Pregunta de negocio actualizada:**
¿Qué combinaciones de género, plataforma y región concentraron mayores ventas globales entre 1980 y 2020, para orientar la selección de género, plataforma y región de un nuevo videojuego?

**Persona que usa la información:** Director de estrategia de producto o gerente de portafolio de una desarrolladora o publisher de videojuegos.

**Decisión que se busca apoyar:** Seleccionar el género, la plataforma y la región objetivo para el próximo lanzamiento, con base en el desempeño histórico de ventas.

**Fuente seleccionada:** `vgsales.csv` (Kaggle, Gregorut, Lic. pública).

**Ajustes derivados de la retroalimentación del Avance 1:** Se recibió retroalimentación formal; por lo que se modifica la pregunta de negocio original, algunos nombres de conceptos y el alcance.

## 2. Lectura

**Contrato de lectura:**

| Elemento | Registro |
|---|---|
| Fuente | Video Game Sales (Kaggle) |
| Autor | Gregorut (basado en VGChartz) |
| Licencia | Pública |
| Archivo | `vgsales.csv` |
| Formato | CSV|
| Ubicación local | `/data/raw/vgsales.csv` |
| Fecha de obtención | Octubre 2026 |
| Estructura esperada | Tabla: una fila por videojuego, 11 columnas |
| Cobertura | 1980–2020, global |
| Unidad de observación | Un videojuego con ventas superiores a 100,000 copias |
| Comprobaciones | `type`, `shape`, `columns`, `head` |
| Advertencias | Faltantes conocidos en `Year` y `Publisher` |

In [1]:
from pathlib import Path

print("Directorio actual:", Path.cwd())
print("\nArchivos en el directorio actual:")

for archivo in Path.cwd().rglob("vgsales.csv"):
    print(archivo)

Directorio actual: /content

Archivos en el directorio actual:
/content/vgsales.csv


In [4]:
from pathlib import Path
import pandas as pd

# Ruta del archivo
ruta = Path("/content/vgsales.csv")  #En el repositorio la ruta es /data/raw/vgsales
assert ruta.exists(), f"No se encontró el archivo: {ruta}"

# Carga sin modificar el origen
datos_crudos = pd.read_csv(ruta, sep=",", encoding="utf-8")
datos_crudos_original = datos_crudos.copy(deep=True)

print(f"Archivo: {ruta}")
print(f"Forma: {datos_crudos.shape}")

Archivo: /content/vgsales.csv
Forma: (16598, 11)


In [5]:
print("Tipo de objeto:", type(datos_crudos))
print("Forma:", datos_crudos.shape)
print("\nColumnas:")
print(datos_crudos.columns.tolist())
print("\nPrimeras 3 filas:")
print(datos_crudos.head(3))
print("\nÚltimas 3 filas:")
print(datos_crudos.tail(3))

Tipo de objeto: <class 'pandas.core.frame.DataFrame'>
Forma: (16598, 11)

Columnas:
['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

Primeras 3 filas:
   Rank               Name Platform    Year     Genre Publisher  NA_Sales  \
0     1         Wii Sports      Wii  2006.0    Sports  Nintendo     41.49   
1     2  Super Mario Bros.      NES  1985.0  Platform  Nintendo     29.08   
2     3     Mario Kart Wii      Wii  2008.0    Racing  Nintendo     15.85   

   EU_Sales  JP_Sales  Other_Sales  Global_Sales  
0     29.02      3.77         8.46         82.74  
1      3.58      6.81         0.77         40.24  
2     12.88      3.79         3.31         35.82  

Últimas 3 filas:
        Rank                                              Name Platform  \
16595  16598  SCORE International Baja 1000: The Official Game      PS2   
16596  16599                                        Know How 2       DS   
16597  16600   

## 3. Perfil inicial

**Unidad de observación:** un videojuego.

**Granularidad:** una fila por videojuego, con ventas por región y globales.

**Llave candidata:** `Name` + `Platform` + `Year` (podría no ser única por relanzamientos).

**Variables numéricas:** `Rank`, `Year`, `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`, `Global_Sales`.

**Variables categóricas:** `Name`, `Platform`, `Genre`, `Publisher`.

**Variables con posibles problemas:** `Year` (faltantes), `Publisher` (faltantes), `Year` como float en lugar de entero.

In [6]:
# Perfil: tipos, faltantes, valores distintos
perfil = pd.DataFrame({
    "tipo_tecnico": datos_crudos.dtypes.astype(str),
    "valores_presentes": datos_crudos.notna().sum(),
    "valores_nulos": datos_crudos.isna().sum(),
    "porcentaje_nulos": (datos_crudos.isna().mean() * 100).round(2),
    "valores_distintos": datos_crudos.nunique(dropna=True),
})
print(perfil)
print("\nDuplicados exactos:", datos_crudos.duplicated().sum())

             tipo_tecnico  valores_presentes  valores_nulos  porcentaje_nulos  \
Rank                int64              16598              0              0.00   
Name               object              16598              0              0.00   
Platform           object              16598              0              0.00   
Year              float64              16327            271              1.63   
Genre              object              16598              0              0.00   
Publisher          object              16540             58              0.35   
NA_Sales          float64              16598              0              0.00   
EU_Sales          float64              16598              0              0.00   
JP_Sales          float64              16598              0              0.00   
Other_Sales       float64              16598              0              0.00   
Global_Sales      float64              16598              0              0.00   

              valores_disti

In [10]:
print("Resumen de variables numéricas\n")
print(datos_crudos.describe().round(2))

Resumen de variables numéricas

           Rank      Year  NA_Sales  EU_Sales  JP_Sales  Other_Sales  \
count  16598.00  16327.00  16598.00  16598.00  16598.00     16598.00   
mean    8300.61   2006.41      0.26      0.15      0.08         0.05   
std     4791.85      5.83      0.82      0.51      0.31         0.19   
min        1.00   1980.00      0.00      0.00      0.00         0.00   
25%     4151.25   2003.00      0.00      0.00      0.00         0.00   
50%     8300.50   2007.00      0.08      0.02      0.00         0.01   
75%    12449.75   2010.00      0.24      0.11      0.04         0.04   
max    16600.00   2020.00     41.49     29.02     10.22        10.57   

       Global_Sales  
count      16598.00  
mean           0.54  
std            1.56  
min            0.01  
25%            0.06  
50%            0.17  
75%            0.47  
max           82.74  


In [16]:
print("   Plataformas:\n")
print(datos_crudos["Platform"].value_counts().head(10))
print("\nTotal plataformas:", datos_crudos["Platform"].nunique())

print("\n   Géneros:\n")
print(datos_crudos["Genre"].value_counts())
print("\nTotal géneros:", datos_crudos["Genre"].nunique())

print("\n   Top 10 Publishers:\n")
print(datos_crudos["Publisher"].value_counts().head(10))
print("\nTotal publishers:", datos_crudos["Publisher"].nunique())

   Plataformas:

Platform
DS      2163
PS2     2161
PS3     1329
Wii     1325
X360    1265
PSP     1213
PS      1196
PC       960
XB       824
GBA      822
Name: count, dtype: int64

Total plataformas: 31

   Géneros:

Genre
Action          3316
Sports          2346
Misc            1739
Role-Playing    1488
Shooter         1310
Adventure       1286
Racing          1249
Platform         886
Simulation       867
Fighting         848
Strategy         681
Puzzle           582
Name: count, dtype: int64

Total géneros: 12

   Top 10 Publishers:

Publisher
Electronic Arts                 1351
Activision                       975
Namco Bandai Games               932
Ubisoft                          921
Konami Digital Entertainment     832
THQ                              715
Nintendo                         703
Sony Computer Entertainment      683
Sega                             639
Take-Two Interactive             413
Name: count, dtype: int64

Total publishers: 578


In [17]:
print("   Top 5 juegos por ventas globales")
print(datos_crudos.nlargest(5, "Global_Sales")[["Name", "Platform", "Year", "Genre", "Global_Sales"]])

print("\n   Años con más lanzamientos:\n")
print(datos_crudos["Year"].value_counts().head(10))

print("\n   Rango temporal:\n")
print(f"Año mínimo: {datos_crudos['Year'].min()}")
print(f"Año máximo: {datos_crudos['Year'].max()}")

   Top 5 juegos por ventas globales
                       Name Platform    Year         Genre  Global_Sales
0                Wii Sports      Wii  2006.0        Sports         82.74
1         Super Mario Bros.      NES  1985.0      Platform         40.24
2            Mario Kart Wii      Wii  2008.0        Racing         35.82
3         Wii Sports Resort      Wii  2009.0        Sports         33.00
4  Pokemon Red/Pokemon Blue       GB  1996.0  Role-Playing         31.37

   Años con más lanzamientos:

Year
2009.0    1431
2008.0    1428
2010.0    1259
2007.0    1202
2011.0    1139
2006.0    1008
2005.0     941
2002.0     829
2003.0     775
2004.0     763
Name: count, dtype: int64

   Rango temporal:

Año mínimo: 1980.0
Año máximo: 2020.0


## 4. Tabla de hallazgos

| # | Columna / registro | Operación que lo detectó | Evidencia | Posible efecto analítico | Aclaración pendiente |
|---|---|---|---|---|---|
| 1 | `Year` | `isna().sum()` | 271 faltantes (1.63%) | No se puede ubicar temporalmente el 1.63% de los juegos; afecta análisis por año | ¿La ausencia es original o error de captura? No hay regla documentada para cambiarlo |
| 2 | `Publisher` | `isna().sum()` | 58 faltantes (0.35%) | No se puede atribuir editorial al 0.35% de los juegos | ¿La ausencia significa "sin publisher" o "no capturado"? |
| 3 | `Year` | `dtypes` | tipo float64 en lugar de entero | Años como 2006.0 dificultan lectura y filtros exactos | Convertir a Int64 sin sustituirlos |
| 4 | `Name` | `nunique()` | 11,493 nombres distintos para 16,598 filas | Hay juegos con mismo nombre en distintas plataformas | Verificar si la llave `Name+Platform+Year` es única |
| 5 | `Rank` | `describe()` | min 1, max 16600, 16,598 valores distintos | Hay 3 rangos no consecutivos (16600 vs 16598 filas) | Revisar si es error de la fuente o simplemente gaps |
| 6 | `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales` | `describe()` | min 0.00 en todas | Un 0 puede ser venta nula real o ausencia disfrazada | Revisar si 0 es valor válido o marcador |
| 7 | `Global_Sales` | `describe()` | min 0.01, max 82.74, sin negativos | Rango coherente con ventas reales | No requiere corrección |

## 5. Reglas de preparación

## Regla 1. Conversión de tipo de `Year`

**Problema:**  
La columna `Year` está definida como `float64` (por ejemplo, `2006.0`) debido a la presencia de valores faltantes. Sin embargo, un año es un valor entero, por lo que su representación como decimal dificulta la aplicación de filtros y la interpretación de los datos.

**Solución:**  
Convertir la columna a `Int64` (tipo entero anulable de Pandas). Posteriormente, verificar que los valores no nulos se mantengan sin cambios, conservando los **16,327 registros con año disponible**.

## Regla 2. No asignar valores estimados o inventados a `Year`

**Problema:**  
Existen **271 valores faltantes** en la columna `Year`. No hay una regla documentada que permita reconstruir o determinar el año de lanzamiento de estos videojuegos.

**Solución:**  
Conservar los valores faltantes (`NaN` o `NA`), sin realizar imputaciones. Excluir estos registros únicamente del subconjunto analítico que requiera información temporal. Por lo tanto, el conteo de valores faltantes se mantiene en **271**.

## Regla 3. No asignar valores estimados o inventados a `Publisher`

**Problema:**  
Existen **58 valores faltantes** en la columna `Publisher`. No se dispone de información suficiente para determinar si corresponden a videojuegos sin publisher asociado o a datos que no fueron capturados.

**Solución:**  
Conservar los valores faltantes, sin realizar imputaciones. Excluir estos registros únicamente de los análisis que requieran la identificación del publisher. Por lo tanto, el conteo de valores faltantes se mantiene en **58**.

## Regla 4. Creación del subconjunto analítico para responder la pregunta de negocio

**Problema:**  
La pregunta de negocio requiere analizar las variables `Genre`, `Platform`, `Region` y `Global_Sales`. Los registros que no contengan información en las columnas necesarias no podrán participar adecuadamente en el análisis.

**Solución:**  
Crear el subconjunto analítico `analisis_videojuegos`, conservando únicamente los registros que tengan valores no nulos en las columnas requeridas: `Genre`, `Platform` y `Global_Sales`. Posteriormente, verificar que el subconjunto no presente valores faltantes en dichas columnas.


**Tratamientos no pertinentes:**

No hay duplicados exactos, así que no se aplica `drop_duplicates`.

No hay problemas de texto ni unidades distintas.

No hay fechas (solo año) ni unidades que requieran conversión.

In [19]:
# Copia de trabajo
datos_trabajo = datos_crudos.copy(deep=True)

# Línea base
linea_base = {
    "filas": datos_trabajo.shape[0],
    "columnas": datos_trabajo.shape[1],
    "duplicados_exactos": datos_trabajo.duplicated().sum(),
    "nulos_Year": datos_trabajo["Year"].isna().sum(),
    "nulos_Publisher": datos_trabajo["Publisher"].isna().sum(),
}
print("Línea base:")
for k, v in linea_base.items(): #recorremos linea_base
    print(f"  {k}: {v}")
print("\n¿Copia independiente?", datos_trabajo is not datos_crudos)

Línea base:
  filas: 16598
  columnas: 11
  duplicados_exactos: 0
  nulos_Year: 271
  nulos_Publisher: 58

¿Copia independiente? True


In [24]:
# Guardar valores originales para comparación
year_original = datos_trabajo["Year"].copy()

# Convertir a Int64 (entero anulable)
datos_trabajo["Year"] = datos_trabajo["Year"].astype("Int64")

# Comprobación
print("Comprobaciones:\n")
print("Tipo original:", year_original.dtype)
print("Tipo nuevo:", datos_trabajo["Year"].dtype)
print("Valores no nulos originales:", year_original.notna().sum())
print("Valores no nulos nuevos:", datos_trabajo["Year"].notna().sum())
print("Faltantes originales:", year_original.isna().sum())
print("Faltantes nuevos:", datos_trabajo["Year"].isna().sum())
print("\nPrimeras 5 filas de Year:\n")
print(datos_trabajo["Year"].head())

Comprobaciones:

Tipo original: Int64
Tipo nuevo: Int64
Valores no nulos originales: 16327
Valores no nulos nuevos: 16327
Faltantes originales: 271
Faltantes nuevos: 271

Primeras 5 filas de Year:

0    2006
1    1985
2    2008
3    2009
4    1996
Name: Year, dtype: Int64


In [25]:
# Columnas indispensables para la pregunta de negocio
columnas_indispensables = ["Genre", "Platform", "Global_Sales"]

# Subconjunto analítico
analisis_videojuegos = datos_trabajo.dropna(
    subset=columnas_indispensables
).copy()

# Impacto
impacto = pd.Series({
    "filas_recibidas": len(datos_trabajo),
    "filas_en_tabla_trabajo": len(datos_trabajo),
    "filas_aptas_para_analisis": len(analisis_videojuegos),
    "filas_fuera_del_analisis": len(datos_trabajo) - len(analisis_videojuegos),
})
print(impacto)
print("\nPorcentaje apto:", round(len(analisis_videojuegos) / len(datos_trabajo) * 100, 2), "%")

filas_recibidas              16598
filas_en_tabla_trabajo       16598
filas_aptas_para_analisis    16598
filas_fuera_del_analisis         0
dtype: int64

Porcentaje apto: 100.0 %


In [27]:
comparacion = pd.DataFrame({
    "Antes (datos crudos)": [
        datos_crudos["Year"].dtype,
        datos_crudos["Year"].isna().sum(),
        datos_crudos["Publisher"].isna().sum(),
        len(datos_crudos),
    ],
    "Después (datos trabajados)": [
        datos_trabajo["Year"].dtype,
        datos_trabajo["Year"].isna().sum(),
        datos_trabajo["Publisher"].isna().sum(),
        len(datos_trabajo),
    ],
}, index=["Tipo Year", "Faltantes Year", "Faltantes Publisher", "Filas"])
print(comparacion)

                    Antes (datos crudos) Después (datos trabajados)
Tipo Year                        float64                      Int64
Faltantes Year                       271                        271
Faltantes Publisher                   58                         58
Filas                              16598                      16598


## 6. Normalización

| Aspecto | ¿Aplica? | Justificación |
|---|---|---|
| Texto (espacios, mayúsculas) | No | Las categorías de `Platform`, `Genre` y `Publisher` ya están en formato consistente, no se observaron variantes como "Nintendo" vs "nintendo" |
| Categorías (equivalencias) | No | No hay códigos ni abreviaturas, los géneros y plataformas son etiquetas completas |
| Fechas | No | Solo hay `Year` como entero, no hay fechas completas que convertir |
| Unidades | No | Las ventas están todas en millones de copias, no hay unidades distintas |
| Tipos | Sí | `Year` se convirtió de float64 a Int64 en la Fase 2 |

**Conclusión:** no se requieren operaciones adicionales de normalización ya que la única conversión necesaria ya se aplicó en esta etapa.

## 7. Validación

**Comprobaciones ejecutables:**

| # | Condición | Uso al que aplica | Acción si falla |
|---|---|---|---|
| 1 | Origen conservado | Garantizar trazabilidad | Revisar la copia de trabajo |
| 2 | Sin duplicados exactos | Evitar inflar conteos | Eliminar duplicados con regla documentada |
| 3 | Año convertido a entero | Análisis por año | Revisar conversión |
| 4 | Categorías completas | Análisis por género y plataforma | Revisar faltantes |
| 5 | Subconjunto sin nulos indispensables | Análisis de ventas por género, plataforma y región | Revisar columnas indispensables |

In [31]:
# Comprobaciones
comprobaciones = pd.Series({
    "origen_conservado": datos_crudos.equals(datos_crudos_original),
    "sin_duplicados_exactos": datos_trabajo.duplicated().sum() == 0,
    "year_convertido_a_entero": str(datos_trabajo["Year"].dtype) == "Int64",
    "genero_plataforma_completos": (
        datos_trabajo["Genre"].notna().all()
        and datos_trabajo["Platform"].notna().all()
    ),
    "subconjunto_sin_nulos_indispensables": (
        analisis_videojuegos[columnas_indispensables].notna().all().all()
    ),
})

print("   Comprobaciones\n")
print(comprobaciones)
assert comprobaciones.all(), "Revisa las reglas que no se cumplieron"
print("\nTodas las comprobaciones pasaron")

   Comprobaciones

origen_conservado                       True
sin_duplicados_exactos                  True
year_convertido_a_entero                True
genero_plataforma_completos             True
subconjunto_sin_nulos_indispensables    True
dtype: bool

Todas las comprobaciones pasaron


## 8. Comparación antes / después

| Aspecto | Antes (crudo) | Después (trabajo) | Cambio |
|---|---|---|---|
| Filas | 16,598 | 16,598 | Sin cambios |
| Columnas | 11 | 11 | Sin cambios |
| Duplicados exactos | 0 | 0 | Sin cambios |
| Tipo de `Year` | float64 | Int64 | Convertido |
| Faltantes en `Year` | 271 | 271 | Conservados |
| Faltantes en `Publisher` | 58 | 58 | Conservados |
| Faltantes en ventas | 0 | 0 | Sin cambios |

**Subconjuntos creados:**

| Subconjunto | Criterio | Filas | Uso |
|---|---|---|---|
| `analisis_videojuegos` | `Genre`, `Platform`, `Global_Sales` presentes | 16,598 | Análisis principal: ventas por género, plataforma y región |
| `analisis_temporal` | `Year` presente | 16,327 | Análisis por año (excluye 271 sin año) |

In [37]:
# Subconjunto adicional para análisis temporal
analisis_temporal = datos_trabajo.dropna(subset=["Year"]).copy()

print("Filas en análisis temporal:", len(analisis_temporal))
print("Filas excluidas por falta de Year:", len(datos_trabajo) - len(analisis_temporal))
print("Porcentaje conservado:", round(len(analisis_temporal) / len(datos_trabajo) * 100, 2), "%")

# Verificar que no haya nulos en Year
print("\nFaltantes en Year:", analisis_temporal["Year"].isna().sum())

Filas en análisis temporal: 16327
Filas excluidas por falta de Year: 271
Porcentaje conservado: 98.37 %

Faltantes en Year: 0


## Cobertura temporal del dataset y alcance del proyecto

El dataset cuenta con una cobertura hasta 2020, sin embargo, el análisis detallado con Pandas mostró que la concentración real de registros termina en 2016, ya que los años 2017 y 2020 contienen casos atípicos (3 y 1 registros, respectivamente) que no deben interpretarse como tendencia. Además, los años 2018 y 2019 no tienen registros.

Por lo tanto, cualquier afirmación sobre tendencias o interpretación para el negocio debe limitarse al periodo **1980-2016**. Los registros atípicos se conservan en el dataset y se reportan como evidencia de la cobertura real.

### Observaciones de la profesora y ajustes realizados

La profesora indicó en el avance 1 que era necesario revisar hasta qué año llegan los datos y documentarlo correctamente, así como ajustar el alcance para no afirmar que el proyecto permite predecir el éxito de un nuevo videojuego ni maximizar ganancias.

Por lo anterior, se ejecutó un conteo de videojuegos por año y se identificó lo siguiente:

- La concentración de registros termina en 2016.
- El año 2017 contiene 3 registros atípicos.
- El año 2020 contiene 1 registro atípico.
- Los años 2018 y 2019 no tienen registros.

Además, se ajustó la pregunta de negocio y el lenguaje del proyecto para sustituir el concepto de *«éxito comercial»* por otros más adecuados al alcance del análisis, como:

- Patrones históricos de desempeño comercial.
- Rendimiento de ventas.
- Toma de decisiones estratégicas.
- Optimización del desempeño comercial.
- Rentabilidad del negocio.
- Retorno de inversión (ROI).

Estos conceptos se utilizan sin realizar afirmaciones sobre el éxito comercial futuro, la capacidad predictiva del análisis o la garantía de resultados.

### Datos que pueden utilizarse

- **16,598 filas del dataset:** para el análisis de ventas por género, plataforma y región.
- **16,327 filas con año válido:** para el análisis temporal, delimitado al periodo 1980-2016 para identificar tendencias.

### Conservación de los datos originales

Se conservan el archivo CSV y el conjunto de datos `datos_crudos` sin modificaciones.

Las principales consideraciones identificadas son:

- **271 videojuegos sin año (1.63%).**
- **58 videojuegos sin publisher (0.35%).**
- **Cobertura temporal:** concentración de registros hasta 2016, con casos aislados en 2017 y 2020.
- **Limitaciones de información financiera:** ausencia de datos de costos, ingresos y marketing.
- **Limitaciones predictivas:** los datos históricos no permiten garantizar predicciones comerciales ni calcular un ROI garantizado.

### Estado actual del avance

Actualmente, este avance se considera **«listo con cautela»** para el análisis de patrones históricos de desempeño comercial por género, plataforma y región, con el propósito de aportar información para la toma de decisiones estratégicas en nuevos proyectos de videojuegos.

In [40]:
# Distribución de juegos por año (últimos 15 años)
conteo_por_anio = datos_crudos["Year"].value_counts().sort_index()
print("     Juegos por año (2010 en adelante):\n")
print(conteo_por_anio[conteo_por_anio.index >= 2010])

print("\n     Resumen de juegos por año\n")
print(f"Total de años con registros: {conteo_por_anio.shape[0]}")
print(f"Año con más juegos: {conteo_por_anio.idxmax()} ({conteo_por_anio.max()} juegos)")
print(f"Año con menos juegos (sin contar 0): {conteo_por_anio[conteo_por_anio > 0].idxmin()}")
print(f"Juegos en 2017: {conteo_por_anio.get(2017, 0)}")
print(f"Juegos en 2018: {conteo_por_anio.get(2018, 0)}")
print(f"Juegos en 2019: {conteo_por_anio.get(2019, 0)}")
print(f"Juegos en 2020: {conteo_por_anio.get(2020, 0)}")

     Juegos por año (2010 en adelante):

Year
2010.0    1259
2011.0    1139
2012.0     657
2013.0     546
2014.0     582
2015.0     614
2016.0     344
2017.0       3
2020.0       1
Name: count, dtype: int64

     Resumen de juegos por año

Total de años con registros: 39
Año con más juegos: 2009.0 (1431 juegos)
Año con menos juegos (sin contar 0): 2020.0
Juegos en 2017: 3
Juegos en 2018: 0
Juegos en 2019: 0
Juegos en 2020: 1
